# Vietnamese AMR Parsing — Fine-Tuning on Colab

Train Qwen2.5 to parse Vietnamese sentences into AMR graphs.

**Pipeline**: SFT → (optional) GRPO → Inference → Smatch scoring.

**Recommended runtime**: T4 GPU (free tier) is enough with LoRA.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone Repository

In [ ]:
!rm -rf /content/ViAMR
# TODO: replace with your repo URL
!git clone https://github.com/YOUR_USERNAME/ViAMR.git /content/ViAMR
%cd /content/ViAMR

## 3. Install Dependencies

In [ ]:
!pip install -q -U \
    'transformers>=4.46' 'trl>=0.12' 'peft>=0.13' 'datasets>=2.21' \
    'accelerate>=1.0' 'penman>=1.3' 'smatch>=1.0.4' \
    'pandas>=2.2' 'underthesea>=6.8' 'wandb>=0.18' 'huggingface_hub>=0.24'

## 4. Validate Dataset

In [ ]:
!bash scripts/prepare_data.sh

## 5. Configure Output Paths

**Option A — Save to Drive**: persistent, slower writes, uses your 15GB quota.
**Option B — Save locally**: faster writes, lost when session ends. Upload to HF Hub at the end (see section 10).

The notebook trains with **LoRA** by default → final checkpoint is only ~200MB, so Drive is fine.

In [ ]:
import os

# --- Option A: Drive (default) ---
DRIVE_ROOT = '/content/drive/MyDrive/ViAMR_outputs'
os.makedirs(DRIVE_ROOT, exist_ok=True)
os.environ['SFT_OUTPUT']   = f'{DRIVE_ROOT}/Qwen-1.5B-SFT'
os.environ['GRPO_OUTPUT']  = f'{DRIVE_ROOT}/Qwen-1.5B-GRPO'
os.environ['RESULTS_FILE'] = f'{DRIVE_ROOT}/results_test.txt'

# --- Option B: local (uncomment to use) ---
# os.makedirs('/content/outputs', exist_ok=True)
# os.environ['SFT_OUTPUT']   = '/content/outputs/Qwen-1.5B-SFT'
# os.environ['GRPO_OUTPUT']  = '/content/outputs/Qwen-1.5B-GRPO'
# os.environ['RESULTS_FILE'] = '/content/outputs/results_test.txt'

print('SFT output  :', os.environ['SFT_OUTPUT'])
print('GRPO output :', os.environ['GRPO_OUTPUT'])
print('Results file:', os.environ['RESULTS_FILE'])
!python colab_storage_helper.py --mode check

## 6. Train SFT (Supervised Fine-Tuning)

LoRA (r=32) on Qwen2.5-1.5B — fits a free T4 and saves ~200 MB instead of 3 GB per checkpoint.

In [ ]:
# Features:
# - eval on data/dev.amr every eval_steps
# - keeps ONLY the best checkpoint (lowest eval_loss), overwrites older best
# - early stopping after 3 evals without improvement
# - auto-resumes if you re-run the cell after a Colab disconnect
!python -m amr.training.sft \
    --dataset1_path data/train.amr \
    --eval_dataset_path data/dev.amr \
    --output_dir "$SFT_OUTPUT" \
    --model_name Qwen/Qwen2.5-1.5B-Instruct \
    --learning_rate 2e-4 \
    --warmup_steps 500 \
    --lr_scheduler_type cosine \
    --logging_steps 10 \
    --per_device_train_batch_size 1 \
    --per_device_eval_batch_size 2 \
    --gradient_accumulation_steps 16 \
    --max_input_length 2048 \
    --num_train_epochs 5 \
    --eval_steps 200 \
    --metric_for_best_model eval_loss \
    --greater_is_better 0 \
    --early_stopping_patience 3 \
    --early_stopping_threshold 0.0 \
    --resume_from_checkpoint auto \
    --use_lora 1 \
    --lora_r 32 \
    --lora_alpha 64 \
    --lora_dropout 0.1

### Storage check after SFT

In [ ]:
# Inspect disk / Drive usage and model size
!python colab_storage_helper.py --mode check

# If intermediate checkpoints piled up, keep only the most recent one
!python colab_storage_helper.py --mode cleanup --output_dir "$SFT_OUTPUT" --keep_last 1

## 7. Inference on the Test Set

In [ ]:
!python -m amr.inference \
    --input_file data/test.amr \
    --output_file "$RESULTS_FILE" \
    --model_name "$SFT_OUTPUT" \
    --my_test 1 \
    --max_retries 5

## 8. Smatch Score

In [ ]:
!python -m amr.scoring \
    --predict_file "$RESULTS_FILE" \
    --gold_file data/test.amr

## 9. (Optional) GRPO — Reinforcement Learning

Starts from the SFT checkpoint and directly optimizes Smatch F1. Expect +2 to +5 points.

In [ ]:
!WANDB_MODE=disabled python -m amr.training.grpo \
    --dataset1_path data/train.amr \
    --output_dir "$GRPO_OUTPUT" \
    --model_name "$SFT_OUTPUT" \
    --learning_rate 5e-6 \
    --warmup_steps 100 \
    --lr_scheduler_type cosine \
    --logging_steps 5 \
    --per_device_train_batch_size 1 \
    --gradient_accumulation_steps 8 \
    --num_generations 4 \
    --max_prompt_length 2048 \
    --max_completion_length 1024 \
    --num_train_epochs 1 \
    --save_steps 200 \
    --save_total_limit 1 \
    --resume_from_checkpoint auto \
    --use_lora 1 \
    --lora_r 32 \
    --lora_alpha 64

## 10. Manage Storage & Share the Model

All cells below are optional — pick whichever applies.

### 10a. Free up space on Drive

In [ ]:
# Current usage:
!python colab_storage_helper.py --mode check

# Drop old checkpoints from SFT and GRPO runs
!python colab_storage_helper.py --mode cleanup --output_dir "$SFT_OUTPUT"  --keep_last 1
!python colab_storage_helper.py --mode cleanup --output_dir "$GRPO_OUTPUT" --keep_last 1

### 10b. Compress the model (~30% smaller)

In [ ]:
!python colab_storage_helper.py --mode compress --output_dir "$SFT_OUTPUT"

### 10c. Upload to Hugging Face Hub (free unlimited storage)

Recommended if Drive is full or you want to share / version the model.

In [ ]:
# 1) Log in once (enter a HF token with write access)
!huggingface-cli login

# 2) Upload. Replace with your own repo id.
!python colab_storage_helper.py --mode upload_hf \
    --output_dir "$SFT_OUTPUT" \
    --repo YOUR_USERNAME/viamr-qwen1.5b-sft

### 10d. Download the model to your local machine

In [ ]:
from google.colab import files
import os, subprocess

sft = os.environ['SFT_OUTPUT']
zip_path = '/content/ViAMR-SFT-model.zip'
subprocess.run(['zip', '-r', zip_path, sft], check=True)
files.download(zip_path)